# 2. Data Understanding

## Sumber dan Bentuk Data

Data yang digunakan adalah hasil ekstraksi sampel piksel citra Sentinel-2A dan disimpan dalam `piksel_s2_JTM.csv`. Notebook memuat berkas tersebut dari folder proyek dan memeriksa keberadaan sejumlah kolom yang diperlukan sebelum analisis dilanjutkan.

Berkas sumber berisi **631.268 baris piksel** dan **19 kolom** pada saat pencatatan. Angka tersebut adalah jumlah baris sebelum penyaringan. Setiap baris merepresentasikan sampel piksel, sedangkan identitas poligon digunakan untuk menghubungkan piksel-piksel yang berasal dari sampel yang sama.

## Kelompok Variabel

| Kelompok | Kolom | Kegunaan |
|---|---|---|
| Identitas sampel | `poligon_id`, `poligon_no`, `sampel_id`, `tile_id` | Menelusuri poligon, sampel, dan tile asal |
| Label | `label`, `label_teks` | Menyimpan kelas dalam bentuk kode dan nama |
| Posisi | `px_row`, `px_col`, `lon`, `lat` | Menyimpan posisi piksel dan koordinat geografis |
| Band spektral | `B02`, `B03`, `B04`, `B08`, `B11` | Menyimpan nilai spektral Sentinel-2A |
| Indeks spektral | `NDVI`, `NDWI`, `MNDWI`, `NDBI` | Merangkum karakteristik spektral yang berkaitan dengan vegetasi, air, dan kawasan terbangun |

Model menggunakan sembilan fitur numerik: lima band (`B02`, `B03`, `B04`, `B08`, `B11`) dan empat indeks (`NDVI`, `NDWI`, `MNDWI`, `NDBI`). Kolom `label_teks` menjadi target klasifikasi.

## Pemeriksaan Data

Notebook memeriksa kolom wajib sebelum memproses data. Jika ada kolom yang hilang, proses dihentikan dengan pesan yang menyebutkan kolom tersebut. Pemeriksaan ini membantu memastikan bahwa dataset sesuai dengan struktur yang dibutuhkan. Untuk menyamakan ruang lingkup analisis dan dashboard, poligon Laut `Laut_0000` sampai `Laut_0018` dikeluarkan dari data analisis karena berada melewati batas wilayah. Baris di CSV sumber tidak dihapus; poligon Laut lainnya tetap dipertahankan.

Satu baris piksel tidak sama dengan satu sampel poligon. Karena itu, jumlah baris piksel yang besar tidak boleh dianggap sebagai jumlah observasi independen untuk evaluasi model. Dashboard menyaring koordinat piksel terhadap batas provinsi terlebih dahulu, lalu menggabungkan piksel berdasarkan `poligon_id` dan memastikan titik rata-rata poligon daratan juga berada di dalam batas sebelum pembagian data. Filter koordinat dashboard mempertahankan sampel Laut yang tidak termasuk dalam daftar pengecualian.

## Output yang Ditampilkan Notebook

Sel berikut memuat ulang dataset, memeriksa kolom wajib, menampilkan ukuran data, dan menghasilkan lima baris pertama langsung dari CSV:

In [1]:
from pathlib import Path

import json

import numpy as np
import pandas as pd
import shapely
from shapely.geometry import shape

dataset_path = next(
    (
        candidate
        for folder in (Path.cwd(), *Path.cwd().parents)
        for candidate in (
            folder / "piksel_s2_JTM.csv",
            folder / "Tugas" / "UTS" / "piksel_s2_JTM.csv",
        )
        if candidate.is_file()
    ),
    None,
)
if dataset_path is None:
    raise FileNotFoundError("Dataset piksel_s2_JTM.csv tidak ditemukan.")

df = pd.read_csv(dataset_path)
boundary_path = dataset_path.with_name("batas_jawa_timur.geojson")
if not boundary_path.is_file():
    raise FileNotFoundError(f"Batas Jawa Timur tidak ditemukan: {boundary_path}")
with boundary_path.open(encoding="utf-8") as file:
    boundary_feature = json.load(file)
if boundary_feature.get("properties", {}).get("adm1_pcode") != "ID35":
    raise ValueError("Berkas geometri bukan batas Provinsi Jawa Timur.")
boundary = shape(boundary_feature["geometry"])
if boundary.is_empty or not boundary.is_valid:
    raise ValueError("Geometri batas Provinsi Jawa Timur tidak valid.")
kolom_wajib = [
    "poligon_id", "label", "label_teks", "lon", "lat",
    "B02", "B03", "B04", "B08", "B11", "NDVI", "NDWI", "MNDWI", "NDBI",
]
kolom_hilang = sorted(set(kolom_wajib) - set(df.columns))
if kolom_hilang:
    raise ValueError(f"Kolom wajib tidak ditemukan: {kolom_hilang}")

poligon_laut_dikecualikan = {f"Laut_{index:04d}" for index in range(19)}
baris_dikecualikan = df["poligon_id"].isin(poligon_laut_dikecualikan)
jumlah_dikecualikan = int(baris_dikecualikan.sum())
coordinates = df[["lon", "lat"]].to_numpy(dtype=np.float64)
finite_coordinates = np.isfinite(coordinates).all(axis=1)
inside_boundary = np.zeros(len(df), dtype=bool)
inside_boundary[finite_coordinates] = shapely.contains_xy(
    boundary,
    coordinates[finite_coordinates, 0],
    coordinates[finite_coordinates, 1],
)
keep_laut = df["label_teks"].eq("Laut").to_numpy()
keep_polygon = ~baris_dikecualikan.to_numpy()
df = df.loc[finite_coordinates & keep_polygon & (inside_boundary | keep_laut)].copy()

print(f"Dataset: {dataset_path}")
print(f"Baris piksel dikecualikan: {jumlah_dikecualikan:,}")
print(f"Jumlah baris (piksel): {len(df):,}")
print(f"Jumlah kolom: {df.shape[1]}")
display(df.head())

Dataset: D:\SEMESTER 5\PSD\TUGAS\Tugas\UTS\piksel_s2_JTM.csv
Baris piksel dikecualikan: 58,000
Jumlah baris (piksel): 562,526
Jumlah kolom: 19


,poligon_id,poligon_no,sampel_id,tile_id,label,label_teks,px_row,px_col,lon,lat,B02,B03,B04,B08,B11,NDVI,NDWI,MNDWI,NDBI
0,Sawah_0000,0,0,x570_y-42,1,Sawah,374,119,114.100652,-8.312431,366.0,532.0,332.0,2903.0,1310.0,0.794745,-0.690247,-0.422367,-0.378115
1,Sawah_0000,0,0,x570_y-42,1,Sawah,375,119,114.100653,-8.312521,457.0,587.0,375.0,3125.0,1448.0,0.785714,-0.683728,-0.423096,-0.366718
2,Sawah_0000,0,0,x570_y-42,1,Sawah,375,120,114.100743,-8.312520,440.0,572.0,349.0,3015.0,1448.0,0.792509,-0.681071,-0.433663,-0.351109
3,Sawah_0000,0,0,x570_y-42,1,Sawah,375,121,114.100834,-8.312519,427.0,552.0,335.0,3020.0,1432.0,0.800298,-0.690929,-0.443548,-0.356694
4,Sawah_0000,0,0,x570_y-42,1,Sawah,375,122,114.100925,-8.312519,465.0,599.0,354.0,3315.0,1432.0,0.807032,-0.693919,-0.410143,-0.396672
